# GraphRAG: Xelsis PDF or AI News CSV

Choose one profile in Section 1. It selects the input file, ontology, all four
LLM prompts, example questions, and output filenames.

| Profile | Input | Ontology and prompts |
|---|---|---|
| Xelsis | data/Xelsis.pdf, pages 5, 7–8, 14–16, 18–21, 28–30 | Machine operation, maintenance, troubleshooting |
| AI news | ai_copyright_dataset.csv | AI copyright and governance |

The pipeline loads source text, extracts entities and relationships, builds
communities, summarizes them, and answers questions from those summaries.

> **Refactored implementation:** The explanatory flow below is preserved from
> the original notebook. The implementation now lives in four reusable modules:
> `graph_rag_schema.py`, `graph_rag_engine.py`, `graph_rag_services.py`, and
> `graph_rag_manager.py`. Notebook cells call those classes instead of repeating
> their implementation.

---
## 0. Install Dependencies

In [1]:
# %pip install -r requirements.txt

---
## 1. Select Profile & Import

Set PROFILE in the next cell to "Xelsis" or "AI news" **before importing src**.
It overrides the active_ontology default in ontology.yaml for this Python process
and selects the same profile in prompt.yaml.

**After changing profiles or editing either YAML file, restart the kernel and
run the notebook from the top.** Already-imported Pydantic models cannot switch
their allowed types safely in place. No YAML files are rewritten by the notebook.

In [2]:
import os
import sys
from pathlib import Path

PROFILE = "AI news"  # Choose "Xelsis" or "AI news"

PROFILES = {
    "Xelsis": {
        "input_type": "pdf",
        "input_file": "data/Xelsis.pdf",
        "output_prefix": "xelsis",
        "questions": [
            "What are the possible causes of watery coffee and the remedy for each cause?",
            "What maintenance does the brew group need, and how often?",
            "When should I replace and activate the AquaClean filter?",
            "Why does coffee come out slowly, and what should I check?",
        ],
    },
    "AI news": {
        "input_type": "csv",
        "input_file": "ai_copyright_dataset.csv",
        "output_prefix": "ai_news",
        "questions": [
            "What are the main legal arguments around AI copyright and training data?",
            "Which companies are involved in AI copyright disputes, and what are their positions?",
            "How are different governments approaching AI governance?",
            "What is the role of fair use in AI copyright disputes?",
        ],
    },
}
if PROFILE not in PROFILES:
    raise ValueError(f"Choose one of {list(PROFILES)}; got {PROFILE!r}")

loaded_schema = sys.modules.get("src.graph_rag_schema")
if loaded_schema is not None and getattr(loaded_schema, "ACTIVE_ONTOLOGY", None) != PROFILE:
    raise RuntimeError("Profile changed after import. Restart the kernel and run from the top.")

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "ontology.yaml").is_file():
    PROJECT_ROOT = PROJECT_ROOT / "causalRAG"
if not (PROJECT_ROOT / "ontology.yaml").is_file():
    raise FileNotFoundError("Start the notebook from causalRAG or its parent directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["GRAPH_RAG_PROFILE"] = PROFILE
PROFILE_CONFIG = PROFILES[PROFILE]
INPUT_TYPE = PROFILE_CONFIG["input_type"]
DATASET_FILE = PROJECT_ROOT / PROFILE_CONFIG["input_file"]

PDF_PAGES = "5, 7-8, 14-16, 18-21, 28-30"  # 1-based PDF pages; None means all pages
PDF_CHUNK_SIZE = 1000
PDF_CHUNK_OVERLAP = 100
MAX_ARTICLES = 10  # CSV only; None means all rows
FORCE_REBUILD = False  # Set True after changing the source, pages, chunks, or YAML

print(f"Profile: {PROFILE} | Input: {INPUT_TYPE.upper()} | File: {DATASET_FILE}")

Profile: AI news | Input: CSV | File: /home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/ai_copyright_dataset.csv


In [3]:
import nest_asyncio
import pandas as pd

from src import (
    ExtractedEntity,
    ExtractedRelationship,
    ExtractionResult,
    GraphRAGExtractor,
    GraphRAGManager,
    GraphRAGQueryEngine,
    GraphRAGSchema,
    GraphRAGService,
    GraphRAGStore,
)

if GraphRAGSchema.ACTIVE_ONTOLOGY != PROFILE:
    raise RuntimeError("Profile mismatch. Restart the kernel and run from the top.")

nest_asyncio.apply()
print(f"✅ Imports ready; ontology and prompts: {GraphRAGSchema.ACTIVE_ONTOLOGY}")

✅ Imports ready; ontology and prompts: AI news


---
## 2. Configuration

Set `USE_QWEN` in the next cell to switch the complete pipeline between the local Qwen model and OpenAI. Each provider uses separate output files so their graphs can be compared safely.

| Provider | Extraction and community summaries | Final query synthesis |
|---|---|---|
| Qwen | `Qwen3.8-27B` through the local LiteLLM endpoint | `Qwen3.8-27B` |
| OpenAI | `gpt-5-nano` | `gpt-5.6-luna` |


In [4]:
USE_QWEN = True  # Set to False to use OpenAI
LLM_PROVIDER = "qwen" if USE_QWEN else "openai"
EXTRACTION_MODEL = "Qwen3.8-27B" if USE_QWEN else "gpt-5-nano"
QUERY_MODEL = "Qwen3.8-27B" if USE_QWEN else "gpt-5.6-luna"
QWEN_BASE_URL = "http://192.168.10.45:4000/v1"

OUTPUT_PREFIX = f"{PROFILE_CONFIG['output_prefix']}_{LLM_PROVIDER}"
CHECKPOINT_FILE = PROJECT_ROOT / f"{OUTPUT_PREFIX}_graph_store.pkl"
GRAPH_DATA_FILE = PROJECT_ROOT / f"{OUTPUT_PREFIX}_graph_data.json"
GRAPH_TEMPLATE_FILE = PROJECT_ROOT / "graph_template.html"
GRAPH_OUTPUT_FILE = PROJECT_ROOT / f"{OUTPUT_PREFIX}_graph.html"

MAX_PATHS_PER_CHUNK = 20    # Max entity-relationship-entity triplet|s extracted per chunk
NUM_WORKERS         = 2     # Lower concurrency reduces API timeouts during structured extraction
MAX_CLUSTER_SIZE    = 10    # Max entities per community cluster (controls how big the clusters should be)
REQUEST_TIMEOUT     = 180.0 # Seconds allowed for each LLM request
REQUEST_MAX_RETRIES = 5     # Retries transient API/network failures with backoff

manager = GraphRAGManager(
    provider=LLM_PROVIDER,
    extraction_model=EXTRACTION_MODEL,
    query_model=QUERY_MODEL,
    qwen_base_url=QWEN_BASE_URL,
    max_paths_per_chunk=MAX_PATHS_PER_CHUNK,
    num_workers=NUM_WORKERS,
    max_cluster_size=MAX_CLUSTER_SIZE,
    request_timeout=REQUEST_TIMEOUT,
    request_max_retries=REQUEST_MAX_RETRIES,
)

EXTRACTION_LLM = manager.extraction_llm
QUERY_LLM = manager.query_llm

print(
    f"✅ Using {LLM_PROVIDER}: {EXTRACTION_LLM.model} for extraction, "
    f"{QUERY_LLM.model} for querying"
)

✅ Using qwen: Qwen3.8-27B for extraction, Qwen3.8-27B for querying


---
## 3. Ontology

The selected profile in ontology.yaml defines allowed entity and relationship
types. The same definitions drive the extraction prompt and Pydantic validation.
Xelsis and AI news have separate vocabularies.

In [5]:
ENTITY_TYPES = GraphRAGSchema.ENTITY_TYPES
RELATION_TYPES = GraphRAGSchema.RELATION_TYPES

print("✅ Ontology:")
print(f"   Entity types:       {ENTITY_TYPES}")
print(f"   Relationship types: {RELATION_TYPES}")

✅ Ontology:
   Entity types:       ('ORGANIZATION', 'PERSON', 'LEGISLATION', 'LEGAL_CASE', 'CONCEPT', 'GOVERNMENT', 'AI_SYSTEM')
   Relationship types: ('FILED_AGAINST', 'DEFENDANT_IN', 'REGULATES', 'ADVOCATES_FOR', 'TRAINED_ON', 'PART_OF', 'REFERENCES', 'OPPOSES')


---
## 4. Extraction Prompt

The selected profile in prompt.yaml supplies the extraction instructions.
Allowed types and their descriptions are inserted from ontology.yaml.
The LLM returns descriptions alongside entities and relationships so community
summaries can retain context, conditions, and qualifications.

In [6]:
KG_TRIPLET_EXTRACT_TMPL = GraphRAGSchema.extraction_prompt()

print("✅ Extraction prompt ready")
print(f"\nPreview (first 300 chars):\n{KG_TRIPLET_EXTRACT_TMPL[:300]}...")

✅ Extraction prompt ready

Preview (first 300 chars):
-Goal-
Given a news article about AI copyright, governance, or intellectual property,
identify the entities and their relationships supported by the article.
Extract up to {max_knowledge_triplets} entity-relation triplets.

-Allowed Entity Types-
- ORGANIZATION: A company, institution, or other orga...


---
## 5. Pydantic Extraction Models

Structured output is validated using the selected ontology:

- ExtractedEntity: name, type, description
- ExtractedRelationship: source, target, relation, description
- ExtractionResult: the entity and relationship lists

EntityType and RelationType are runtime Literal types loaded from YAML.
Pydantic rejects labels outside the selected profile.

In [7]:
print("✅ Pydantic extraction models:")
print("  ", ExtractedEntity.__name__)
print("  ", ExtractedRelationship.__name__)
print("  ", ExtractionResult.__name__)

✅ Pydantic extraction models:
   ExtractedEntity
   ExtractedRelationship
   ExtractionResult


---
## 6. GraphRAGExtractor

This is the core extraction component. It sends each text chunk to the LLM
with our ontology-constrained prompt, parses the response, and stores the
extracted entities and relationships as structured objects on each node.

### Why build a custom extractor?

LlamaIndex's built-in `SchemaLLMPathExtractor` extracts entity/relationship labels
but **drops descriptions**. By building our own extractor:

- Every `EntityNode` carries a `entity_description` property
- Every `Relation` carries a `relationship_description` property
- These flow through to community summaries, making them far richer

The extractor runs **asynchronously** with `num_workers=2` — processing 2 chunks
in parallel while limiting pressure on long structured-output API requests.


In [8]:
kg_extractor = manager.extractor

print(f"✅ {type(kg_extractor).__name__} ready")
print(f"   Parallel workers: {kg_extractor.num_workers}")
print(f"   Maximum paths per chunk: {kg_extractor.max_paths_per_chunk}")

✅ GraphRAGExtractor ready
   Parallel workers: 2
   Maximum paths per chunk: 20


---
## 7. GraphRAGStore

`GraphRAGStore` extends LlamaIndex's `SimplePropertyGraphStore` with two additional
capabilities: **community detection** and **community summary generation**.

By bundling these into the store itself, the pipeline stays clean — after building
the index you just call `graph_store.build_communities()` and everything is handled.

### How community detection works here

1. Convert the property graph to a NetworkX graph
2. Run hierarchical Leiden to find entity clusters
3. For each cluster, collect all entities (+ descriptions) and relationships (+ descriptions)
4. Ask the LLM to write a briefing note for each cluster

The descriptions captured during extraction make these briefings significantly
richer than if we'd only stored bare labels.


In [9]:
# The manager creates this store when a graph is built or loaded.
print(f"✅ Graph store class ready: {GraphRAGStore.__name__}")

✅ Graph store class ready: GraphRAGStore


---
## 8. GraphRAGQueryEngine

The query engine uses a two-step approach:

1. **Per-community answering** — ask the LLM to answer the question from each
   community summary independently. If a summary isn't relevant, the LLM says so
   and we skip it. This avoids polluting the final answer with irrelevant content.

2. **Aggregation** — combine all relevant partial answers into one final,
   non-redundant response using `QUERY_LLM` (the stronger model).


In [10]:
# The query engine is instantiated after a graph has been built or loaded.
print(f"✅ Query engine class ready: {GraphRAGQueryEngine.__name__}")

✅ Query engine class ready: GraphRAGQueryEngine


---
## 9. Inspect the Selected Input

Preview CSV rows for AI news, or the page/chunk settings for Xelsis.
The next section performs the actual loading. The PDF loader handles text-based
PDFs; pages without extractable text are skipped and reported.

In [11]:
if not DATASET_FILE.is_file():
    raise FileNotFoundError(f"Input file not found: {DATASET_FILE}")

if INPUT_TYPE == "csv":
    df = pd.read_csv(DATASET_FILE)
    print(f"CSV articles available: {len(df)} | Article limit: {MAX_ARTICLES}")
    display(df.head())
else:
    print(f"PDF: {DATASET_FILE.name}")
    print(f"Pages: {PDF_PAGES if PDF_PAGES is not None else 'all'}")
    print(f"Chunk size: {PDF_CHUNK_SIZE} tokens | Overlap: {PDF_CHUNK_OVERLAP}")

CSV articles available: 13 | Article limit: 10


,query,title,snippet,source,date,url,type,full_text,video_id,status
0,AI intellectual property,Generative AI: Navigating intellectual property,Fully AI-generated content is ineligible for c...,Nixon Peabody,"Sep 17, 2025",https://www.nixonpeabody.com/insights/articles...,article,Generative AI is transforming creative and tec...,NaN,success
1,AI intellectual property,Artificial Intelligence and Intellectual Property,AI inventions present the current patent syste...,World Intellectual Property Organization (WIPO),NaN,https://www.wipo.int/en/web/frontier-technolog...,article,Artificial Intelligence and Intellectual Prope...,NaN,success
2,AI intellectual property,"AI, Copyright, and the Law: The Ongoing Battle...","In the past year, courts, legislators, and reg...",University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article,By: Negar Bondari\nArtificial intelligence (AI...,NaN,success
3,AI intellectual property,AI Created It—But Do You Own It? IP Issues Exp...,AI-generated content raises complex legal ques...,DarrowEverett LLP,"Jul 7, 2025",https://darroweverett.com/ai-and-the-law-who-o...,article,Legal Insights\nAs artificial intelligence (AI...,NaN,success
4,AI intellectual property,Copyright and Artificial Intelligence | U.S. C...,Copyright and Artificial Intelligence analyzes...,Copyright Office (.gov),NaN,https://www.copyright.gov/ai/,article,Copyright and Artificial Intelligence\nSince l...,NaN,success


---
## 10. Load Documents or PDF Chunks

AI news loads CSV rows through manager.load_documents().
Xelsis extracts the selected PDF pages and splits each page into token chunks
through manager.load_pdf_documents(). Chunks retain file, page, and chunk
metadata. PDF overlap stays within a page; it does not bridge page boundaries.

In [12]:
if GraphRAGSchema.ACTIVE_ONTOLOGY != PROFILE:
    raise RuntimeError("Profile mismatch. Restart the kernel and run from the top.")

if INPUT_TYPE == "pdf":
    nodes = manager.load_pdf_documents(
        DATASET_FILE,
        pages=PDF_PAGES,
        chunk_size=PDF_CHUNK_SIZE,
        chunk_overlap=PDF_CHUNK_OVERLAP,
    )
elif INPUT_TYPE == "csv":
    nodes = manager.load_documents(DATASET_FILE, max_articles=MAX_ARTICLES)
else:
    raise ValueError(f"Unsupported input type: {INPUT_TYPE}")

if not nodes:
    raise ValueError("The selected input produced no documents.")
print(f"✅ Loaded {len(nodes)} input nodes for {PROFILE}")

Created 10 LlamaIndex documents
✅ Loaded 10 input nodes for AI news


In [13]:
SAMPLE_INDEX = 0  # Choose any loaded document/chunk index
if not 0 <= SAMPLE_INDEX < len(nodes):
    raise IndexError(f"SAMPLE_INDEX must be between 0 and {len(nodes) - 1}")
print(nodes[SAMPLE_INDEX].metadata)
print(nodes[SAMPLE_INDEX].get_content())

{'title': 'Generative AI: Navigating intellectual property', 'source': 'Nixon Peabody', 'date': 'Sep 17, 2025'}
Generative AI is transforming creative and technology-driven industries at an unprecedented pace. While AI technologies have great potential to unlock new business opportunities and efficiencies, they also raise questions about authorship, ownership, and protecting human creativity.
These questions underscore a central legal dilemma: current intellectual property laws are struggling to keep pace with the unique challenges posed by AI-generated content. Traditional legal frameworks and precedents for copyright, patent, and trademark protection were conceived with human creators in mind, leaving uncertainty about how to protect, license, and enforce rights in works generated by, or with the assistance of, AI. Nixon Peabody serves as a forward-thinking partner for businesses exploring AI and intellectual property opportunities.
How does AI affect intellectual property frameworks

---
## 11. Build the Knowledge Graph 

Now we wire everything together and run the extraction pipeline.

`PropertyGraphIndex` handles the full workflow:
1. Passes each chunk to `GraphRAGExtractor`
2. The extractor calls the LLM with our ontology-constrained prompt
3. Parsed entities and relationships are stored in `GraphRAGStore`

-> This is the most time-consuming step!


Checkpoint, JSON, and HTML filenames include both the dataset profile and LLM
provider, so Xelsis and AI news outputs are kept separate.

Set FORCE_REBUILD = True in Section 1 after changing the source, PDF pages,
chunk settings, ontology, or prompts. Otherwise an existing checkpoint for the
selected profile/provider is reused. Restart the kernel after YAML changes.
Older notebook checkpoint filenames are not automatically reused.

In [14]:
CHECKPOINT_FILE.exists()

False

In [15]:
REBUILD_GRAPH = FORCE_REBUILD or not CHECKPOINT_FILE.exists()

if REBUILD_GRAPH:
    # Community detection is kept for Section 12 so the original flow remains clear.
    graph_store = manager.build_knowledge_graph(
        documents=nodes,
        build_communities=False,
    )
else:
    graph_store = manager.load_knowledge_graph(CHECKPOINT_FILE)

Building knowledge graph; LLM extraction may take several minutes...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
# Optional extra LLM call on one loaded document/chunk.
RUN_EXTRACTION_TEST = False
if RUN_EXTRACTION_TEST:
    extraction_test = await manager.atest_extraction(document_index=SAMPLE_INDEX)

In [17]:
entities_by_type = manager.print_unique_entities()


AI_SYSTEM (14)
  AI Image and Text Generation Platforms
  AI Systems
  ChatGPT
  Creativity Machine
  GPT series
  Generative AI
  Generative AI Tools
  Generative AI models
  Large Language Model (LLM)
  Large Language Models
  Machine learning systems
  Midjourney image generation software
  Saturn brand protection platform
  Stable Diffusion

CONCEPT (41)
  AI Copyright Regulations
  AI Inventions
  AI-generated intellectual property
  AI-generated works
  Artificial Intelligence (AI)
  Compendium of U.S. Copyright Office Practices
  Copyright
  Copyright Infrastructure
  Copyright Infringement
  Copyright Law
  Copyrightability of AI outputs
  Copyrightability of Outputs
  Copyrighted Material
  Deepfakes
  Designs
  Digital Replicas
  Digital replicas
  Fair Use
  Generative AI Training
  Human Authorship
  Human authorship
  Human creativity
  Human inventorship
  IP enforcement
  Intellectual Property
  Intellectual Property (IP)
  Intellectual Property (IP) rights
  Internatio

In [18]:
# Choose an entity that actually exists in the selected graph.
ENTITY_TO_INSPECT = next(
    (name for names in entities_by_type.values() for name in names),
    None,
)
entity_details = (
    manager.inspect_entity(ENTITY_TO_INSPECT)
    if ENTITY_TO_INSPECT is not None else None
)

Node: 'AI Image and Text Generation Platforms'  label='AI_SYSTEM'
Source: University of South Florida
Title: Copyright and Generative AI - AI Tools and Resources

=== Article text ===
Copyright is a form of intellectual property protection provided by the laws of the United States (title 17, U.S. Code) to the creators of "original works of authorship." Copyright grants creators/authors the ability to control the use of the work upon creation for the life of the author +70 years, and gives to authors certain exclusive rights:
Learn more:
Your ability to use material generated by AI tools in projects, presentations, and publications is affected by a number factors:
Material created by generative AI tools does not currently receive copyright protections in the United States. However, generative AI tools can create 'new' material that infringes on existing copyrights. Using this material in a project, presentation, or publication would have to abide by copyright law.
Some policies, confere

---
## 12. Build Communities & Generate Summaries

Community summaries use the selected profile's community_summary template in
prompt.yaml. Xelsis summaries emphasize maintenance and troubleshooting;
AI news summaries emphasize copyright and governance.

In [19]:
if REBUILD_GRAPH:
    summaries = graph_store.build_communities(
        summary_llm=EXTRACTION_LLM,
        max_cluster_size=MAX_CLUSTER_SIZE,
    )
    manager.save_knowledge_graph(CHECKPOINT_FILE)
else:
    summaries = graph_store.get_community_summaries()

print(f"\n✅ {len(summaries)} community summaries ready for querying")

Running community detection...
Graph has 132 nodes, 148 edges
Found 40 communities


/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/.venv/lib/python3.12/site-packages/graspologic/partition/leiden.py:607: UserWarning: Leiden partitions do not contain all nodes from the input graph because input graph contained isolate nodes.
  warnings.warn(


  Community 0: The US Copyright Office and the DC Circuit are central to the current legal landscape, where the US ...
  Community 1: The U.S. Copyright Office has released a multi-part report analyzing key AI copyright issues, includ...
  Community 2: The legal landscape surrounding AI-generated intellectual property is currently defined by significa...
  Community 3: AI developers are currently navigating a complex regulatory landscape where they rely on the legal d...
  Community 4: The U.S. Copyright Office has released a report addressing key legal and policy issues at the inters...
  Community 5: The World Intellectual Property Organization (WIPO) is central to the global governance landscape, h...
  Community 6: The U.S. Copyright Office, led by Register Shira Perlmutter, is advancing a regulatory framework tha...
  Community 7: The New York Times has filed a lawsuit against OpenAI and Microsoft, alleging that the defendants’ G...
  Community 8: China has established a regulator

---
## 13. Visualize the Knowledge Graph

Export the selected graph as JSON and render the HTML visualization at
GRAPH_OUTPUT_FILE.

In [20]:
manager.visualize(
    graph_data_file=GRAPH_DATA_FILE,
    template_file=GRAPH_TEMPLATE_FILE,
    output_file=GRAPH_OUTPUT_FILE,
)

Graph data exported to '/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/ai_news_qwen_graph_data.json'
Nodes: 132 | Edges: 148 | Communities: 40
Visualization saved to '/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/ai_news_qwen_graph.html'


PosixPath('/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/ai_news_qwen_graph.html')

---
## 14. Query the System

Example questions follow the selected input profile. Answers use community
summaries and the matching community_answer and aggregation prompts from
prompt.yaml.

In [ ]:
query_engine = GraphRAGQueryEngine(
    graph_store=graph_store,
    community_llm=EXTRACTION_LLM,
    llm=QUERY_LLM,
)

print("✅ Query engine ready")

In [ ]:
q1 = PROFILE_CONFIG["questions"][0]
print(f"Query: {q1}")
print("=" * 70)
print(query_engine.custom_query(q1))

In [ ]:
q2 = PROFILE_CONFIG["questions"][1]
print(f"Query: {q2}")
print("=" * 70)
print(query_engine.custom_query(q2))

In [ ]:
q3 = PROFILE_CONFIG["questions"][2]
print(f"Query: {q3}")
print("=" * 70)
print(query_engine.custom_query(q3))

In [ ]:
# Replace this with your own question about the selected dataset.
your_question = PROFILE_CONFIG["questions"][3]
print(f"Query: {your_question}")
print("=" * 70)
print(query_engine.custom_query(your_question))

In [ ]:
CHECKPOINT_FILE.exists()

---
## 15. Pipeline Reference

| Component | Purpose |
|---|---|
| PROFILE | Select input, vocabulary, prompts, questions, and output names |
| ontology.yaml | Allowed entity and relationship types |
| prompt.yaml | Extraction, summary, answering, and synthesis instructions |
| GraphRAGManager | Load PDF/CSV, build, persist, inspect, and query |
| GraphRAGStore | Entity graph and community summaries |
| D3.js visualization | Interactive graph in the selected HTML output |